In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid, GridSearchCV, PredefinedSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


In [2]:
data_folder = Path.cwd().parent / "data" / "processed"

df = pd.read_csv(
    data_folder / "weather_dataset_2024_2026.csv", 
    parse_dates=["time"]
)

In [3]:

df["hour_of_day"] = df["time"].dt.hour
df["hour_sin"] = np.sin(2 * np.pi * df["hour_of_day"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour_of_day"] / 24)

df = df.sort_values("time").set_index("time")
df = df.reset_index()

train = df[df["time"].dt.year == 2024]
val = df[df["time"].dt.year == 2025]
test = df[df["time"].dt.year == 2026]

features = [
    "temperature_ECMWF",
    "temperature_GFS",
    "temperature_ICON_EU",
    "hour_sin",
    "hour_cos",
]

target = "actual_temperature"

X_train = train[features]
y_train = train[target]

X_val = val[features]
y_val = val[target]

X_test = test[features]
y_test = test[target]

In [4]:
def model_mae_and_rmse(pred, y, model_name):
    print(f"\n{model_name}")
    print("MAE:", mean_absolute_error(y, pred))
    print("RMSE:", root_mean_squared_error(y, pred))

**Predicting temperature**

In [5]:
lr = LinearRegression()

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_val)

model_mae_and_rmse(lr_pred, y_val, "Linear regression")


Linear regression
MAE: 0.9265575786339546
RMSE: 1.199126026952906


In [6]:

X = sm.add_constant(train[features])
y = train[target]

ols = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 24})
print(ols.summary())

X_val_c = sm.add_constant(X_val, has_constant="add")
ols_pred = ols.predict(X_val_c)

model_mae_and_rmse(ols_pred, y_val, "Linear regression wiht HAC errors")

                            OLS Regression Results                            
Dep. Variable:     actual_temperature   R-squared:                       0.919
Model:                            OLS   Adj. R-squared:                  0.919
Method:                 Least Squares   F-statistic:                     1732.
Date:                Wed, 25 Nov 2026   Prob (F-statistic):               0.00
Time:                        16:24:09   Log-Likelihood:                -3489.4
No. Observations:                2208   AIC:                             6991.
Df Residuals:                    2202   BIC:                             7025.
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.0141    

In [7]:

alphas = np.logspace(-3, 3, 25)

rows = []
for a in alphas:
    m = make_pipeline(StandardScaler(), Ridge(alpha=a))
    m.fit(X_train, y_train)
    p = m.predict(X_val)
    rows.append({"alpha": a,
                 "val_MAE": mean_absolute_error(y_val, p),
                 "val_RMSE": root_mean_squared_error(y_val, p)})

res = pd.DataFrame(rows)
best_alpha = res.loc[res["val_MAE"].idxmin(), "alpha"]
print("best alpha:", best_alpha)

ridge = make_pipeline(StandardScaler(), Ridge(alpha=best_alpha))
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_val)

model_mae_and_rmse(ridge_pred, y_val, "Ridge regression")


best alpha: 0.001

Ridge regression
MAE: 0.9265575996576196
RMSE: 1.1991261305126297


In [8]:

temp_forecasts = ["temperature_ECMWF", "temperature_GFS", "temperature_ICON_EU"]

def add_interactions(X):
    X = X.copy()
    for c in temp_forecasts:
        X[f"{c}_x_hsin"] = X[c] * X["hour_sin"]
        X[f"{c}_x_hcos"] = X[c] * X["hour_cos"]
    return X

Xtr_i, Xva_i = add_interactions(X_train), add_interactions(X_val)

best_a = None
best_mae = 99
best_model = None
for a in np.logspace(-3, 3, 25):
    m = make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(Xtr_i, y_train)
    mae = mean_absolute_error(y_val, m.predict(Xva_i))
    if mae < best_mae:
        best_a, best_mae, best_model = a, mae, m

ridge_inter_pred = best_model.predict(Xva_i)

model_mae_and_rmse(ridge_inter_pred, y_val, "Ridge regression with interaction effects")


Ridge regression with interaction effects
MAE: 0.9251502167638004
RMSE: 1.2019618622726507


In [13]:
Xtr_final_i = add_interactions(pd.concat([X_train, X_val]))

ridge_final_m = make_pipeline(
    StandardScaler(), 
    Ridge(alpha=best_a)
).fit(Xtr_final_i, pd.concat([y_train, y_val]))

ridge_pred = best_model.predict(add_interactions(X_test))
model_mae_and_rmse(ridge_pred, y_test, "Ridge regression with interaction effects on test data")


Ridge regression with interaction effects on test data
MAE: 0.8574129741763573
RMSE: 1.1076064542356119


In [9]:

X_all = pd.concat([X_train, X_val])
y_all = pd.concat([y_train, y_val])
split = PredefinedSplit(np.r_[np.full(len(X_train), -1), np.zeros(len(X_val))])

param_grid = {
    "n_estimators":     [250, 500, 1000],
    "max_depth":        [3, 5, 8, 12, None],
    "min_samples_leaf": [5, 10, 20, 50],
    "max_features":     [0.33, 0.5, 1.0],
}

search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,    cv=split,
    scoring="neg_mean_absolute_error",
    refit=False,
    n_jobs=1,
    verbose=1,
)

search.fit(X_all, y_all)

print("best params:", search.best_params_)
print("best val MAE:", -search.best_score_)


Fitting 1 folds for each of 180 candidates, totalling 180 fits
best params: {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 10, 'n_estimators': 500}
best val MAE: 0.9792035207208789


In [10]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1, **search.best_params_)
rf.fit(X_all, y_all)  

rf_pred = rf.predict(X_test)

model_mae_and_rmse(rf_pred, y_test, "Random forest on test data")


Random forest on test data
MAE: 0.850829401494
RMSE: 1.1171074665097447
